In [ ]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

In [ ]:
!pip install scikit-learn==1.2.2
import sklearn
import pypfopt
print(sklearn.__version__)
print(pypfopt.__version__)

1.2.2
1.5.5


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy

In [ ]:
!pip install yfinance==0.2.40
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

yfinance: pandas_datareader support is deprecated & semi-broken so will be removed in a future verison. Just use yfinance.


In [ ]:
def modify_date(date):
    return date[:10]

In [ ]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

In [ ]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

In [ ]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

In [ ]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

In [ ]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

In [ ]:
def output(weight):
    if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

    elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

    stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX[0])
    injected_result["oex"]=S2*OEX
    return injected_result

In [ ]:
# Preselection parameters set up
method = "K_means_multiple"          # or avg or anyother method
stock_type = "S&P100"
end_swan = "Swan10D"         #equals the last swan before the swan we are going to predict
start_swan="Swan5"
start = dt(2023,7,28)
end = dt(2024,11,1)
selected_stock_number = 22     #number of stock preselected
start_swan_date = dt(2024,11,1)
end_swan_date = dt(2024,11,6)
w1 = 0.17
w2 = 0.17
MA_days = 30
N = 7 # This parameter is set to help classification method like K-means

# Step 1 - Preselection

In [ ]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel("/content/All_SwanBeta100.xlsx",index_col=0)
else:
    portfolio_beta = pd.read_excel("All_SwanBeta500.xlsx",index_col=0)

In [ ]:
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

In [ ]:
portfolio_beta= portfolio_beta.loc[:,"Swan1":end_swan]
portfolio_beta

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
Ticker,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.935350
GM,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,1.734848
KMI,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,NaN
GOOG,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,1.184058
GOOGL,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,1.201377
...,...,...,...,...,...,...,...,...,...,...
META,NaN,NaN,NaN,NaN,NaN,NaN,1.142586,1.183531,1.011852,2.289337
PYPL,NaN,NaN,NaN,NaN,NaN,NaN,1.440542,0.580067,0.989242,2.093559
CRM,NaN,NaN,NaN,NaN,0.955665,1.505787,0.782578,1.189409,0.860818,1.626909


In [ ]:
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan

In [ ]:
selected_corr

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
Swan1,1.0,-0.003532,0.112309,0.050640,0.001797,0.102327,0.079934,0.077152,0.038135,-0.097569
Swan2,NaN,1.000000,0.548025,0.335659,0.410765,0.390310,0.033350,0.307687,0.037435,0.484681
Swan3,NaN,NaN,1.000000,0.158060,0.260501,0.440204,0.249559,0.228378,0.106342,0.347335
Swan4,NaN,NaN,NaN,1.000000,-0.022423,0.156316,0.155139,0.233628,0.093825,0.303871
Swan5,NaN,NaN,NaN,NaN,1.000000,0.370946,0.254273,0.219081,0.566815,0.156276
Swan6,NaN,NaN,NaN,NaN,NaN,1.000000,0.226079,0.447314,0.362923,0.247842
Swan7,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.138652,0.188918,-0.065778
Swan8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.207563,0.123789
Swan9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,-0.040622
Swan10D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [ ]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
#portfolio_beta["corr_beta"]=selected_beata@weight_swan
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

In [ ]:
portfolio_beta

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
Ticker,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.935350
GM,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,1.734848
KMI,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,NaN
GOOG,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,1.184058
GOOGL,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,1.201377
...,...,...,...,...,...,...,...,...,...,...
META,NaN,NaN,NaN,NaN,NaN,NaN,1.142586,1.183531,1.011852,2.289337
PYPL,NaN,NaN,NaN,NaN,NaN,NaN,1.440542,0.580067,0.989242,2.093559
CRM,NaN,NaN,NaN,NaN,0.955665,1.505787,0.782578,1.189409,0.860818,1.626909


In [ ]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1,n_init=10)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
    else:
        preselected_stock = first_group


# Use results from multiple clusters
elif method=="K_means_multiple":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    seed_dict={}

    # Run 1000 trials with davies bouldin index
    for r in range(1,1001):
      Kmean = KMeans(n_clusters=N,random_state=r,n_init=10)
      Kmean.fit(selected_beta)
      labels = Kmean.labels_
      # Evaluate the davies-bouldin score
      dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      seed_dict[r]=dbscore
      sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

    # Put all stocks into a stock list
    stock_list=[]
    max_length=0
    for index in range(0,5):
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10)
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      Kmean.labels_
      selected_beta["Group"]=Kmean.labels_
      res = ct.mean(axis=1)
      res [res<0]=0
      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
      if first_group.shape[0] <13:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group
      if preselected_stock.shape[0]>max_length:
        max_length=preselected_stock.shape[0]
      stock_list.extend(preselected_stock.index.values)
    print(stock_list)

    # Count the times each stock appear
    values,counts=np.unique(stock_list,return_counts=True)
    count_dict={}
    preselected_stock=[]
    for t in range(0,len(values)):
      count_dict[values[t]]=counts[t]
    sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
    print(sorted_count)

    d = {'stock':[], 'frequency':[]}
    df = pd.DataFrame(data=d)
    for item in sorted_count:
      df.loc[len(df.index)] = [item[0],item[1]]
    df.to_csv('stock_frequency.csv')

    for p in range(0,min(22,len(sorted_count))):
      if sorted_count[p][1]>=1:
        preselected_stock.append(sorted_count[p][0])

    preselected_stock = sorted(preselected_stock)
    selected_beta=selected_beta.loc[preselected_stock]
    selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #selected_beta["avg"]=selected_beta["avg"].mean()
    preselected_stock = selected_beta
    print(preselected_stock)


<ipython-input-90-ce698ba3cc8a>:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-90-ce698ba3cc8a>:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-90-ce698ba3cc8a>:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_b

['MDLZ', 'SBUX', 'EXC', 'PEP', 'KO', 'PG', 'DUK', 'ABT', 'MCD', 'NEE', 'VZ', 'AMT', 'SO', 'ABT', 'MCD', 'SO', 'AMT', 'NEE', 'PEP', 'EXC', 'MDLZ', 'PG', 'SBUX', 'VZ', 'DUK', 'KO', 'CL', 'NEE', 'ABT', 'VZ', 'SO', 'KO', 'PEP', 'AMT', 'DUK', 'PG', 'SBUX', 'EXC', 'MDLZ', 'MCD', 'ABT', 'AMT', 'SO', 'MCD', 'PEP', 'MDLZ', 'DUK', 'LLY', 'VZ', 'WMT', 'KO', 'SBUX', 'NEE', 'MRK', 'EXC', 'PG', 'NEE', 'CL', 'ABT', 'DUK', 'PEP', 'AMT', 'MCD', 'PG', 'MDLZ', 'MRK', 'EXC', 'WMT', 'LLY', 'KO', 'SO', 'VZ', 'SBUX', 'AMT', 'AMGN', 'VZ', 'MRK', 'NEE', 'SBUX', 'PEP', 'MO', 'DUK', 'PG', 'MCD', 'LLY', 'SO', 'KO', 'WMT', 'JNJ', 'CL', 'MDLZ', 'BMY', 'EXC', 'NEE', 'KO', 'MCD', 'SBUX', 'AMT', 'PEP', 'VZ', 'PG', 'MDLZ', 'EXC', 'SO', 'DUK', 'CL', 'SO', 'SBUX', 'MCD', 'AMT', 'NEE', 'PEP', 'VZ', 'MDLZ', 'PG', 'EXC', 'KO', 'DUK', 'CL', 'SO', 'SBUX', 'MCD', 'AMT', 'NEE', 'PEP', 'VZ', 'MDLZ', 'PG', 'EXC', 'KO', 'DUK', 'CL', 'DUK', 'PG', 'SO', 'AMT', 'EXC', 'VZ', 'NEE', 'MCD', 'PEP', 'MRK', 'KO', 'LLY', 'MDLZ', 'SBUX']
[('

<ipython-input-90-ce698ba3cc8a>:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-90-ce698ba3cc8a>:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-90-ce698ba3cc8a>:72: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_b

In [ ]:
res

array([1.10652171, 0.72374011, 0.6922777 , 1.34084528, 0.77249889,
       1.55979288, 0.39785361])

In [ ]:
preselected_stock.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21 entries, ABT to WMT
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Swan6    21 non-null     float64
 1   Swan7    21 non-null     float64
 2   Swan8    21 non-null     float64
 3   Swan9    21 non-null     float64
 4   Swan10D  21 non-null     float64
 5   Group    21 non-null     int32  
 6   avg      21 non-null     float64
dtypes: float64(6), int32(1)
memory usage: 1.2+ KB


In [ ]:
#portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan]*weight

In [ ]:
preselected_stock["PR"]=1-preselected_stock["avg"]
preselected_stock.index

Index(['ABT', 'AMGN', 'AMT', 'BMY', 'CL', 'DUK', 'EXC', 'JNJ', 'KO', 'LLY',
       'MCD', 'MDLZ', 'MO', 'MRK', 'NEE', 'PEP', 'PG', 'SBUX', 'SO', 'VZ',
       'WMT'],
      dtype='object', name='Ticker')

# Step 2 - Download data and adjuste price based on Swan Beta

In [ ]:
#next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
#selected_stock_list = pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
#swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

In [ ]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
print(selected_stock_list)

['ABT' 'AMGN' 'AMT' 'BMY' 'CL' 'DUK' 'EXC' 'JNJ' 'KO' 'LLY' 'MCD' 'MDLZ'
 'MO' 'MRK' 'NEE' 'PEP' 'PG' 'SBUX' 'SO' 'VZ' 'WMT']


In [ ]:
#download data from yahoo finance
yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Adj Close"]


[*********************100%%**********************]  21 of 21 completed


In [ ]:
yahooDataFrame

Ticker,ABT,AMGN,AMT,BMY,CL,DUK,EXC,JNJ,KO,LLY,...,MDLZ,MO,MRK,NEE,PEP,PG,SBUX,SO,VZ,WMT
Date,,,,,,,,,,,,,,,,,,,,,
2023-07-28,109.915901,227.623642,181.125046,57.867027,73.503075,88.497070,39.703518,167.909180,60.128529,454.445831,...,73.225174,40.806568,102.891647,70.201515,183.278214,151.653519,98.220207,68.942818,31.237297,52.407055
2023-07-31,108.550858,225.485779,182.409424,58.602024,74.125153,88.762543,39.855858,161.220901,59.599236,450.550415,...,72.337166,40.743771,103.191597,70.635162,180.533539,151.546875,98.530624,68.933281,31.283192,52.390671
2023-08-01,107.770813,223.530899,183.300812,57.980103,74.494514,87.368820,39.484535,162.529694,59.445251,449.103241,...,72.551849,40.528484,101.866035,69.546234,179.917191,151.498367,98.229912,67.923210,30.686535,52.144875
2023-08-02,105.908508,221.884201,182.706543,58.187408,75.369316,87.729095,39.513096,163.511292,59.628101,450.481049,...,73.527672,39.981289,102.272400,69.295692,181.352112,151.828033,99.073875,67.627815,30.585558,52.180923
2023-08-03,104.855461,222.163437,178.038727,57.697411,74.436195,85.681168,38.560974,164.213806,59.320145,445.207855,...,73.469124,39.739082,102.301445,67.551491,180.206085,152.225571,99.267876,65.388474,30.346901,52.194035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-25,114.220001,316.980011,222.800003,51.919998,95.610001,117.209999,40.070000,160.880005,66.919998,892.700012,...,69.230003,49.709999,103.980003,81.430000,171.789993,168.220001,97.309998,91.879997,41.380001,82.510002
2024-10-28,114.070000,317.170013,222.220001,52.639999,95.739998,117.059998,40.230000,161.600006,66.669998,895.590027,...,69.540001,50.250000,104.230003,82.870003,169.839996,169.300003,98.019997,91.790001,41.619999,82.750000
2024-10-29,113.400002,316.000000,212.910004,52.500000,94.720001,114.410004,39.549999,160.089996,65.559998,903.580017,...,69.150002,49.950001,103.730003,79.559998,167.500000,167.169998,97.389999,89.820000,41.330002,81.699997


In [ ]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
#swan_to_predict=100
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

In [ ]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv("trainingMarketdrop.csv")

In [ ]:
market_drop.to_csv("market_drop.csv")

In [ ]:
market_drop

,0
0,0.0000
1,0.0028
2,0.0055
3,0.0083
4,0.0110
...,...
314,0.0000
315,0.0000
316,0.0000
317,0.0000


In [ ]:
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

In [ ]:
yahooDataFrame


Ticker,ABT,AMGN,AMT,BMY,CL,DUK,EXC,JNJ,KO,LLY,...,MDLZ,MO,MRK,NEE,PEP,PG,SBUX,SO,VZ,WMT
Date,,,,,,,,,,,,,,,,,,,,,
2023-07-28,109.915901,227.623642,181.125046,57.867027,73.503075,88.497070,39.703518,167.909180,60.128529,454.445831,...,73.225174,40.806568,102.891647,70.201515,183.278214,151.653519,98.220207,68.942818,31.237297,52.407055
2023-07-31,108.322573,225.268391,182.159018,58.524959,74.018856,88.700170,39.803341,160.995642,59.524692,450.335640,...,72.248003,40.694407,103.106281,70.542038,180.337986,151.330773,98.374313,68.889999,31.242731,52.331615
2023-08-01,107.325620,223.107588,182.806540,57.830330,74.284677,87.248226,39.382338,162.083628,59.299204,448.682717,...,72.376188,40.432032,101.700604,69.366131,179.534381,151.074018,97.923809,67.839439,30.608573,52.029416
2023-08-02,105.248281,221.250093,181.963060,57.960580,75.048933,87.546357,39.358760,162.834072,59.407025,449.844492,...,73.259018,39.837699,102.021752,69.024880,180.769810,151.186258,98.607970,67.501946,30.468294,52.006565
2023-08-03,103.989161,221.321995,177.078562,57.399327,74.016848,85.444637,38.361361,163.312430,59.028665,444.374102,...,73.113360,39.549936,101.969167,67.201617,179.439236,151.372797,98.649201,65.227183,30.192704,51.962900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-25,114.220001,316.980011,222.800003,51.919998,95.610001,117.209999,40.070000,160.880005,66.919998,892.700012,...,69.230003,49.709999,103.980003,81.430000,171.789993,168.220001,97.309998,91.879997,41.380001,82.510002
2024-10-28,114.070000,317.170013,222.220001,52.639999,95.739998,117.059998,40.230000,161.600006,66.669998,895.590027,...,69.540001,50.250000,104.230003,82.870003,169.839996,169.300003,98.019997,91.790001,41.619999,82.750000
2024-10-29,113.400002,316.000000,212.910004,52.500000,94.720001,114.410004,39.549999,160.089996,65.559998,903.580017,...,69.150002,49.950001,103.730003,79.559998,167.500000,167.169998,97.389999,89.820000,41.330002,81.699997


In [ ]:
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

array(['ABT', 'AMGN', 'AMT', 'BMY', 'CL', 'DUK', 'EXC', 'JNJ', 'KO',
       'LLY', 'MCD', 'MDLZ', 'MO', 'MRK', 'NEE', 'PEP', 'PG', 'SBUX',
       'SO', 'VZ', 'WMT'], dtype=object)

# Step3 - MVO

In [ ]:
#Set the paramter for MVO process
Portfolio_Date=end

In [ ]:
#data cleaning - filling missing data in yahooDataFrame
data_cleaning(yahooDataFrame)

In [ ]:
#calculated rf
#bond_yield = quandl.get('FRED/DGS3MO',api_key='X4EQzCpMfeaTCTDthRH_')
#bond_yield = quandl.get('FRED/DGS3MO')
#bond_yield=np.log(bond_yield/100+1)
#date=list(map(str,bond_yield.index.values))
#date=list(map(modify_date,date))
#week=pd.to_datetime(date).strftime("%W").values
#week_1=np.roll(week,1)
#week_difference=np.where(week!=week_1)
#week_difference=np.delete(week_difference,0,None)
#rf_week=bond_yield.iloc[week_difference-1]

In [ ]:
# Calculated R for 1 year and 0.5 year
today_price = yahooDataFrame.iloc[-MA_days:,:]
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]
half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
half_year_ago_date = half_year_ago_data.index.values[-1]
one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)



In [ ]:
# Calculate MA
MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
MA_Data=MA_Data.loc[one_year_ago_date:,:]
MA_now = MA_Data.iloc[-1,:]
MA_one_year_ago = MA_Data.iloc[0,:]
MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

In [ ]:
# Calculate daily return
daily_return = yahooDataFrame.pct_change().dropna()
log_daily_return = np.log(1+daily_return)

# Get week end stock price data
stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
# Calculate weekly return

weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)

# Get month end stock price data
stock_date = yahooDataFrame.index.values
month=pd.to_datetime(stock_date).strftime("%m").values
month_1=np.roll(month,1)
month_difference=np.where(month!=month_1)
month_difference=np.delete(month_difference,0,None)
# Calculate monthly return
monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
log_monthly_return = np.log(1+monthly_return)

# Get semi-month end stock price data
semi_month = []
for i in range(1,len(month_difference)):
    semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
    semi_month.append(month_difference[i]-1)
# Calculate semi-monthly return
semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
log_semi_monthly_return = np.log(1+semi_monthly_return)

In [ ]:
# Calculated Max Dropdown for each stock
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
mdd=[]
for column in stock_return_one_year.columns:
    mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
mar=R_mv_one_year/mdd.iloc[:,0]

In [ ]:
#calculate weekly sd for each stock
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

#Calculate garch_std sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
judge = one_year_monthly_return.shape[0]
for stock in one_year_monthly_return.columns:
    res=res_all[stock]
    h= het_arch(res)
    if judge < 100 or h ==0:
        garch_std = weekly_std

    else:#build Garch Model
        std = []
        if judge <=200:
            p = 0
            q = 6
        else:
            p = 1
            q = 1

        std.append(fit_garch(one_year_monthly_return[stock],p,q))
        garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

#Calculate monthly sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

#Calculate semi_monthly sd for each stock
one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [ ]:
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

In [ ]:
# Read OPT-data
# optdata = pd.read_excel("/content/optdata_SP100_2024_10_01.xlsx",index_col=0,sheet_name = 0)
optdata = pd.read_csv("optdata_SP100_20241101.csv",index_col=0)
P_current = MA_now
P_fwd = optdata.P_fwd.loc[P_current.index]
R_fwd=P_fwd/P_current-1
R_60 = P_current/MA_one_year_ago - 1
R_60_half = P_current/MA_half_year_ago - 1
R_MA_1ago = R_mv_one_year
R_MA_halfago = R_mv_half_year
Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
for i in R_fwd.index:
    if R_fwd[i] < 0.5*R_60[i]:
        Rfwd_adjusted[i] = R_60_half[i]
    elif R_fwd[i] > 1.3*R_60[i]:
        if 1.3*R_60[i] > 0.7*R_fwd[i]:
            Rfwd_adjusted[i] = 1.3*R_60[i]
        else:
            Rfwd_adjusted[i] = 0.7*R_fwd[i]
    else:
        Rfwd_adjusted[i] = R_fwd[i]
Rfwd_adjusted[Rfwd_adjusted<0]=0

In [ ]:
price_now = today_price.iloc[-1,:]
optimizer = optdata.loc[P_current.index,:]
FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
preselected_stock = preselected_stock.loc[selected_stock_list,:]
preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
preselected_stock["std_f"]=weekly_std.values
preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
preselected_stock.to_csv("preselected_Stocks_info.csv")
p=FFC.MA_now
EPS_fwd=optimizer.EPS_fwd
EPS_ttm=optimizer.EPS_ttm
fall=FFC.Fall_MDD
FFC.to_csv("WEDataBank.csv")

PE=p/EPS_fwd
PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
PEG[PEG<0]=2000


In [ ]:
preselected_stock

,Swan6,Swan7,Swan8,Swan9,Swan10D,Group,avg,PR,Rfwd_adjusted,std_f,SR,PR_SR
Ticker,,,,,,,,,,,,
ABT,0.311079,1.073150,0.431921,0.941927,0.997309,1,0.751077,0.248923,0.097490,0.204789,0.476052,0.171748
AMGN,0.109292,0.843475,0.659977,0.526728,-0.417885,1,0.344317,0.655683,0.204874,0.251047,0.816080,0.592325
AMT,0.050751,0.583639,-0.336659,0.882155,1.271482,6,0.490274,0.509726,0.284716,0.275655,1.032870,0.518036
BMY,-0.398965,1.296441,1.085742,0.931988,-0.566874,1,0.469667,0.530333,0.117217,0.295894,0.396146,0.333792
CL,-0.045657,0.784583,0.569564,0.632697,0.619560,1,0.512150,0.487850,0.165690,0.147104,1.126340,0.517752
DUK,-0.322793,0.233701,-0.356250,1.165454,0.534699,6,0.250962,0.749038,0.238313,0.154740,1.540086,0.929557
EXC,0.263389,0.605246,-0.123438,1.254518,0.353268,6,0.470597,0.529403,0.124912,0.176567,0.707449,0.445281
JNJ,0.360346,0.668742,0.555245,0.789805,0.120880,1,0.499004,0.500996,0.085295,0.156965,0.543404,0.369314
KO,0.325968,0.614677,-0.049526,1.161009,0.181359,6,0.446697,0.553303,0.193743,0.131579,1.472445,0.671401


In [ ]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
E_2 = Rfwd_adjusted/fall
E_3 = Rfwd_adjusted/PEG
E_4 = Rfwd_adjusted
std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


# Optimization

In [ ]:
w0 = 1/selected_stock_number*np.ones(selected_stock_number)
uplim_ini = np.zeros(selected_stock_number)
uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
for i in uplim3.index:
    if i in ["SPY","IWD"]:
        uplim3.loc[i,:]=0
low_lim =  np.zeros(selected_stock_number)
multi=lambda x: x.T @cov@x

In [ ]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
#ef.add_constraint(lambda w: w@E_2== 12)
raw_weights = ef.max_sharpe(0.01)
cleaned_weights = ef.clean_weights()
print(cleaned_weights)
ef.portfolio_performance(verbose=True)


OrderedDict([('ABT', 0.0), ('AMGN', 0.0), ('AMT', 0.0), ('BMY', 0.0), ('CL', 0.0), ('DUK', 0.17), ('EXC', 0.0), ('JNJ', 0.0), ('KO', 0.1632), ('LLY', 0.1568), ('MCD', 0.0), ('MDLZ', 0.0), ('MO', 0.17), ('MRK', 0.0), ('NEE', 0.0), ('PEP', 0.0), ('PG', 0.0), ('SBUX', 0.0), ('SO', 0.17), ('VZ', 0.0), ('WMT', 0.17)])
Expected annual return: 2.9%
Annual volatility: 9.0%
Sharpe Ratio: 0.21


/usr/local/lib/python3.10/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


(0.029016194634876785, 0.09046003531645466, 0.21021652897163684)

In [ ]:
weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

In [ ]:
sd1 = np.sqrt(weight1@cov@weight1.T)
R1 = weight1.T@E_2/100
sharpe_ratio1 = (R1-0.01)/sd1
print("new_sharpe:",sharpe_ratio1)

new_sharpe: 0.21021652896770335


weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

sd2 = np.sqrt(weight2@cov@weight2.T)
R2 = weight2.T@E_2/100
sharpe_ratio2 = (R2-0.01)/sd2
print("old_sharpe:",sharpe_ratio2)

In [ ]:
weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
weight = weight[weight.weight >=0.03]
weight.weight = weight.weight/sum(weight.weight)
#weight = pd.read_csv("weights_SP100.csv",index_col=0)
weight.to_csv("weights.csv")

See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference


In [ ]:
if stock_type == "S&P100":
    OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

elif stock_type == "S&P500":
    OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  6 of 6 completed


In [ ]:
stock_price.to_csv("tradeinfo.csv")
stock_price

Ticker,DUK,KO,LLY,MO,SO,WMT
Date,,,,,,
2024-11-01,112.680000,65.010002,818.929993,53.869999,88.540001,82.190002
2024-11-04,112.519997,65.110001,806.140015,53.880001,88.120003,82.449997
2024-11-05,114.080002,65.370003,806.059998,54.189999,88.699997,83.680000


In [ ]:
print(stock_price.head(1))
weight


Ticker         DUK         KO         LLY         MO         SO        WMT
Date                                                                      
2024-11-01  112.68  65.010002  818.929993  53.869999  88.540001  82.190002


,weight
DUK,0.1700
KO,0.1632
LLY,0.1568
MO,0.1700
SO,0.1700
WMT,0.1700


In [ ]:
rest=1
extra=0
for i, row in weight.iterrows():
  if row['weight']>0.17:
    rest=rest-row['weight']
    extra=extra+(row['weight']-0.17)
    row['weight']=0.17

for i,row in weight.iterrows():
  if row['weight']<0.165:
    row['weight']=row['weight']+row['weight']/rest*extra
    a=1

weight.to_csv("weights.csv")
weight

,weight
DUK,0.1700
KO,0.1632
LLY,0.1568
MO,0.1700
SO,0.1700
WMT,0.1700


In [ ]:
# weight = pd.read_csv("weights100.csv",index_col=0)
injected_result = pd.DataFrame(index = stock_price.index)
S1 = np.array(1378291.96242148*weight.T)/np.array(stock_price.iloc[0,:]).T
weight

,weight
DUK,0.1700
KO,0.1632
LLY,0.1568
MO,0.1700
SO,0.1700
WMT,0.1700


In [ ]:
portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
S1 = np.array(1378291.96242148*weight.T)/np.array(stock_price.iloc[0,:]).T
injected_result["portfolio_value"]=stock_price@S1.T
S2 = 1285642.65872896/float(OEX[0])
injected_result["oex"]=S2*OEX
injected_result.to_csv("Injected_result.csv")


<ipython-input-164-06fc4e17db10>:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 1285642.65872896/float(OEX[0])


In [ ]:
injected_result

,portfolio_value,oex
Date,,
2024-11-01,1.378292e+06,1.285643e+06
2024-11-04,1.374603e+06,1.279890e+06
2024-11-05,1.385115e+06,1.294769e+06


In [ ]:
allocation=pd.read_excel('allocation weight.xlsx',index_col='stock selected')
dfMVOstock=deepcopy(weight)
dfMVOstock['PR_SR']=''
dfMVOstock['SR']=''
for i in range(len(weight)):
    dfMVOstock.iloc[i,1]=preselected_stock.loc[weight.index[i],'PR_SR']
    dfMVOstock.iloc[i,2]=preselected_stock.loc[weight.index[i],'SR']
dfMVOstock_sort=dfMVOstock.sort_values(by='PR_SR',ascending=False)
postallocationweight=post_allocation(dfMVOstock_sort)
postallocationweight


<ipython-input-11-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<ipython-input-11-2b9c13413cb0>:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfMVOweight.iloc[i,5]=allocationweight[0]
<ipython-input-11-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<i

,weight,PR_SR,SR,equal,proportion,adjusted proportion,direct
SO,0.1700,1.051999,1.838979,0.166667,0.211215,0.17,0.17
DUK,0.1700,0.929557,1.540086,0.166667,0.186631,0.17,0.17
WMT,0.1700,0.887369,2.206222,0.166667,0.178161,0.17,0.17
MO,0.1700,0.721911,1.61936,0.166667,0.144941,0.166944,0.17
LLY,0.1568,0.718472,0.749761,0.166667,0.144251,0.166253,0.17
KO,0.1632,0.671401,1.472445,0.166667,0.1348,0.156803,0.15


In [ ]:
#draw type E graph for postallocation
MVOweight=postallocationweight['weight']
equalweight=postallocationweight['equal']
proportionweight=postallocationweight['adjusted proportion']
directweight=postallocationweight['direct']
output(MVOweight).to_csv('no_post_allocation_result.csv')
output(equalweight).to_csv('equal_weight_result.csv')
output(proportionweight).to_csv('proportional_weight_result.csv')
output(directweight).to_csv('direct_weight_result.csv')

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  6 of 6 completed
<ipython-input-12-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  6 of 6 completed
<ipython-input-12-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  6 of 6 comple

#